In [2]:
import os
import re
import sqlite3
import pandas as pd

# =====================================================================
# 1. FILE LOAD INITIALIZATION (Restores missing 'dfs' scope)
# =====================================================================
base_path = r"C:\Users\mahal\Desktop\MyWorkSpace\Projects\railpulse_sql_analysis\data_files"

print("Loading raw GTFS text datasets...")
calendar_df = pd.read_csv(os.path.join(base_path, "calendar.txt"))
cal_dates_df = pd.read_csv(os.path.join(base_path, "calendar_dates.txt"))
routes_df = pd.read_csv(os.path.join(base_path, "routes.txt"))
stop_times_df = pd.read_csv(os.path.join(base_path, "stop_times.txt"))
stops_df = pd.read_csv(os.path.join(base_path, "stops.txt"))
trips_df = pd.read_csv(os.path.join(base_path, "trips.txt"))
transfers_df = pd.read_csv(os.path.join(base_path, "transfers.txt"))

# Pack loaded sheets into global workspace container
dfs = {
    'routes': routes_df,
    'calendar': calendar_df,
    'calendar_dates': cal_dates_df,
    'trips': trips_df,
    'stop_times': stop_times_df,
    'stops': stops_df,
    'transfers': transfers_df
}

# =====================================================================
# 2. PREPROCESSING HELPERS
# =====================================================================
def drop_missing_columns(df):
    return df.dropna(axis=1, how='all')

def convert_to_datetime(df, column_names: list):
    df = df.copy()
    for col in column_names:
        df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')
    return df

def _clean_single_gtfs_time(time_str):
    if pd.isna(time_str):
        return None
    match = re.match(r"^(\d+):(\d{2}):(\d{2})", str(time_str).strip())
    if not match:
        return None
    hours, minutes, seconds = map(int, match.groups())
    return f"{hours % 24:02d}:{minutes:02d}:{seconds:02d}"

def clean_gtfs_time(df, column_names: list):
    df = df.copy()
    for col in column_names:
        df[col] = df[col].apply(_clean_single_gtfs_time)
    return df

def preprocessing_dfs(df):
    df = df.copy()
    df = drop_missing_columns(df)
    date_columns = [col for col in df.columns if 'date' in col.lower()]
    time_columns = [col for col in df.columns if 'time' in col.lower()]
    
    if date_columns:
        df = convert_to_datetime(df, date_columns)
    if time_columns:
        df = clean_gtfs_time(df, time_columns)
    return df

# =====================================================================
# 3. DATABASE INGESTION LOOPER
# =====================================================================
def create_and_populate_db(db_name="railpulse.db", cleaned_dfs=None):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = OFF;")
    
    schema_sql = """
    DROP TABLE IF EXISTS transfers;
    DROP TABLE IF EXISTS stop_times;
    DROP TABLE IF EXISTS trips;
    DROP TABLE IF EXISTS stops;
    DROP TABLE IF EXISTS calendar_dates;
    DROP TABLE IF EXISTS calendar;
    DROP TABLE IF EXISTS routes;

    CREATE TABLE routes (
        route_id TEXT PRIMARY KEY,
        agency_id TEXT,
        route_short_name TEXT,
        route_long_name TEXT,
        route_type INTEGER NOT NULL,
        route_color TEXT,
        route_text_color TEXT
    );

    CREATE TABLE calendar (
        service_id TEXT PRIMARY KEY,
        monday INTEGER NOT NULL CHECK (monday IN (0, 1)),
        tuesday INTEGER NOT NULL CHECK (tuesday IN (0, 1)),
        wednesday INTEGER NOT NULL CHECK (wednesday IN (0, 1)),
        thursday INTEGER NOT NULL CHECK (thursday IN (0, 1)),
        friday INTEGER NOT NULL CHECK (friday IN (0, 1)),
        saturday INTEGER NOT NULL CHECK (saturday IN (0, 1)),
        sunday INTEGER NOT NULL CHECK (sunday IN (0, 1)),
        start_date TEXT NOT NULL,
        end_date TEXT NOT NULL
    );

    CREATE TABLE calendar_dates (
        service_id TEXT,
        date TEXT NOT NULL,
        exception_type INTEGER NOT NULL CHECK (exception_type IN (1, 2)),
        PRIMARY KEY (service_id, date),
        FOREIGN KEY (service_id) REFERENCES calendar(service_id) ON DELETE CASCADE
    );

    CREATE TABLE stops (
        stop_id TEXT PRIMARY KEY,
        stop_name TEXT NOT NULL,
        stop_desc TEXT,
        stop_lat REAL,
        stop_lon REAL,
        location_type INTEGER DEFAULT 0,
        parent_station TEXT,
        platform_code TEXT,
        FOREIGN KEY (parent_station) REFERENCES stops(stop_id) ON DELETE SET NULL
    );

    CREATE TABLE trips (
        trip_id TEXT PRIMARY KEY,
        route_id TEXT NOT NULL,
        service_id TEXT NOT NULL,
        trip_headsign TEXT,
        trip_short_name TEXT,
        block_id INTEGER,
        bikes_allowed INTEGER,
        FOREIGN KEY (route_id) REFERENCES routes(route_id) ON DELETE CASCADE,
        FOREIGN KEY (service_id) REFERENCES calendar(service_id) ON DELETE CASCADE
    );

    CREATE TABLE stop_times (
        trip_id TEXT NOT NULL,
        arrival_time TEXT,
        departure_time TEXT,
        stop_id TEXT NOT NULL,
        stop_sequence INTEGER NOT NULL,
        pickup_type INTEGER DEFAULT 0,
        drop_off_type INTEGER DEFAULT 0,
        PRIMARY KEY (trip_id, stop_sequence),
        FOREIGN KEY (trip_id) REFERENCES trips(trip_id) ON DELETE CASCADE,
        FOREIGN KEY (stop_id) REFERENCES stops(stop_id) ON DELETE CASCADE
    );

    CREATE TABLE transfers (
        from_stop_id TEXT NOT NULL,
        to_stop_id TEXT NOT NULL,
        transfer_type INTEGER NOT NULL,
        min_transfer_time INTEGER,
        from_trip_id TEXT,
        to_trip_id TEXT,
        FOREIGN KEY (from_stop_id) REFERENCES stops(stop_id) ON DELETE CASCADE,
        FOREIGN KEY (to_stop_id) REFERENCES stops(stop_id) ON DELETE CASCADE,
        FOREIGN KEY (from_trip_id) REFERENCES trips(trip_id) ON DELETE CASCADE,
        FOREIGN KEY (to_trip_id) REFERENCES trips(trip_id) ON DELETE CASCADE
    );

    CREATE INDEX idx_stop_times_stop_id ON stop_times(stop_id);
    CREATE INDEX idx_trips_route_id ON trips(route_id);
    CREATE INDEX idx_transfers_stops ON transfers(from_stop_id, to_stop_id);
    """
    
    print("Building schema architecture and optimization tracking layers...")
    cursor.executescript(schema_sql)
    conn.commit()

    if cleaned_dfs: 
        insertion_order = ['routes', 'calendar', 'calendar_dates', 'stops', 'trips', 'stop_times', 'transfers']
        print("\nStreaming cleaned datasets directly into SQLite records...")
        
        for table in insertion_order:
            df = cleaned_dfs.get(table)
            if df is not None:
                working_df = df.copy()

                
                for col in working_df.columns:
                    if pd.api.types.is_datetime64_any_dtype(working_df[col]):
                        working_df[col] = working_df[col].dt.strftime('%Y-%m-%d')

                if table == 'calendar_dates':
                    working_df = working_df.drop_duplicates(subset=['service_id', 'date'])
                elif table == 'stop_times':
                    working_df = working_df.drop_duplicates(subset=['trip_id', 'stop_sequence'])
                elif table in ['routes', 'calendar', 'stops', 'trips']:
                    pk_col = f"{table[:-1] if table.endswith('s') else table}_id"
                    if pk_col in working_df.columns:
                        working_df = working_df.drop_duplicates(subset=[pk_col])

                cursor.execute(f"PRAGMA table_info({table});")
                schema_cols = [row[1] for row in cursor.fetchall()]
                valid_df_cols = [c for c in working_df.columns if c in schema_cols]
                
                upload_df = working_df[valid_df_cols]
                upload_df.to_sql(table, conn, if_exists='append', index=False, chunksize=75000)
                print(f" -> Table '{table}': Successfully loaded {len(upload_df)} rows.")
            else:
                print(f" -> Skipped '{table}': Missing dataset resource map link.")

    cursor.execute("PRAGMA foreign_keys = ON;")
    conn.close()
    print(f"\nDatabase deployment complete. Workspace file updated as '{db_name}'.")

# =====================================================================
# 4. EXECUTION FLOW PIPELINE
# =====================================================================
if __name__ == '__main__':
    print("\n[STEP 1/2] Processing staging data parameters...")
    for name, df in dfs.items():
        if df is not None:
            dfs[name] = preprocessing_dfs(df)
            print(f" -> Transformation validation complete for '{name}'.")

    print("\n[STEP 2/2] Synchronizing structure database streams...")
    create_and_populate_db("railpulse.db", cleaned_dfs=dfs)


Loading raw GTFS text datasets...

[STEP 1/2] Processing staging data parameters...
 -> Transformation validation complete for 'routes'.
 -> Transformation validation complete for 'calendar'.
 -> Transformation validation complete for 'calendar_dates'.
 -> Transformation validation complete for 'trips'.
 -> Transformation validation complete for 'stop_times'.
 -> Transformation validation complete for 'stops'.
 -> Transformation validation complete for 'transfers'.

[STEP 2/2] Synchronizing structure database streams...
Building schema architecture and optimization tracking layers...

Streaming cleaned datasets directly into SQLite records...
 -> Table 'routes': Successfully loaded 1801 rows.
 -> Table 'calendar': Successfully loaded 51593 rows.
 -> Table 'calendar_dates': Successfully loaded 51593 rows.
 -> Table 'stops': Successfully loaded 2895 rows.
 -> Table 'trips': Successfully loaded 134809 rows.
 -> Table 'stop_times': Successfully loaded 2165519 rows.
 -> Table 'transfers': S